<a href="https://colab.research.google.com/github/ingridevv/data-engineering-zoomcamp/blob/main/week_6_batch/1_pyspark_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [4]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [5]:
# Initialize spark session
spark = SparkSession.builder \
    .appName('spark_intro') \
    .getOrCreate()

## Start Tunnel to Access SparkUI with ngrok

In [ ]:
from pyngrok import ngrok, conf
import getpass

conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

··········


 * ngrok tunnel "https://senaida-interlobular-franklyn.ngrok-free.dev" -> "http://127.0.0.1:4040"


## Download Yellow Taxi Data, Read it with Spark

In [7]:
# Download file directly
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-05 11:21:20--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-05T12%3A06%3A26Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-05T11%3A05%3A44Z&ske=2026-03-05T12%3A06%3A26Z&sks=b&skv=2018-11-09&sig=rawtBEMsf6GUjvJlhaYhqmaQjNPKNVlUu7u3Gvfcqeg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjcxMzI4MCwibmJmIjoxNzcyNzA5NjgwLCJwYXRoIjoi

In [8]:
# Read the file (Spark unzip automatically)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

df.count()

11908468

In [9]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', IntegerType(), True)])

In [10]:
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
+-----------------+--------------------+-------------------+-------------------+

In [11]:
# Read the first 101 rows with pandas (zcat unzip first)
!zcat fhvhv_tripdata_2021-01.csv | head -n 101 > head.csv

In [12]:
!wc -l head.csv

101 head.csv


In [14]:
import pandas as pd

df_pandas = pd.read_csv('head.csv')

In [15]:
df_pandas.dtypes

,0
hvfhs_license_num,object
dispatching_base_num,object
pickup_datetime,object
dropoff_datetime,object
PULocationID,int64
DOLocationID,int64
SR_Flag,float64


In [ ]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [16]:
from pyspark.sql import types

In [17]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)]
)

In [18]:
# Use our defined schema
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

In [19]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

### Save as parquet files
*Note: Remember that Spark is a cluster with multiple executors that can handle one file at a time*
We should cut this one large file and split into partitions -> df.repartition()

In [20]:
df = df.repartition(24)

In [21]:
df.write.parquet('fhvhv/2021/01/')

In [22]:
!ls -lh fhvhv/2021/01/

total 226M
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00000-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00001-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00002-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00003-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00004-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00005-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00006-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00007-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:23 part-00008-b2714285-d4ed-4551-8fff-27b144d70dbb-c000.snappy.parquet
-rw-r--r

In [23]:
df = spark.read.parquet('fhvhv/2021/01/')

In [24]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



### Spark Actions vs Transformations
- **Transformations** (lazy Evaluation): Selections, filtering, joins, groupBy...
- **Actions** (immediate execution): show, take, head, write...

In [28]:
from pyspark.sql import functions as F

In [35]:
# UDFs

def base_id(num):
  location_id = int(num)
  if location_id % 2:
    return f'A-{num}10y'
  else:
    return f'B-{num}10x'

In [37]:
base_id_udf = F.udf(base_id, returnType=types.StringType())

In [39]:
# Transformation: adding two columns
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
  .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
  .withColumn('base_id', base_id_udf(df.PULocationID)) \
  .select('base_id', 'PULocationID', 'DOLocationID', 'pickup_date', 'dropoff_date') \
  .show()

+--------+------------+------------+-----------+------------+
| base_id|PULocationID|DOLocationID|pickup_date|dropoff_date|
+--------+------------+------------+-----------+------------+
| A-9710y|          97|          25| 2021-01-10|  2021-01-10|
|B-13810x|         138|         265| 2021-01-08|  2021-01-08|
| B-5010x|          50|         163| 2021-01-01|  2021-01-01|
|A-16310y|         163|          79| 2021-01-15|  2021-01-15|
| A-4710y|          47|          74| 2021-01-12|  2021-01-12|
| B-4810x|          48|          95| 2021-01-05|  2021-01-05|
| A-6310y|          63|          77| 2021-01-02|  2021-01-02|
|B-23810x|         238|          41| 2021-01-06|  2021-01-06|
| A-6310y|          63|         244| 2021-01-02|  2021-01-02|
|B-21010x|         210|         165| 2021-01-24|  2021-01-24|
|A-11310y|         113|         143| 2021-01-16|  2021-01-16|
| A-9110y|          91|          89| 2021-01-28|  2021-01-28|
|A-23110y|         231|         148| 2021-01-01|  2021-01-01|
|B-22810

In [26]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID')\
.filter(df.hvfhs_license_num == 'HV0003') \
.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|
|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|
|2021-01-02 00:34:43|2021-01-02 00:45:38|          63|          77|
|2021-01-02 16:20:11|2021-01-02 16:56:36|          63|         244|
|2021-01-24 16:00:53|2021-01-24 16:07:40|         210|         165|
|2021-01-16 19:35:17|2021-01-16 19:50:20|         113|         143|
|2021-01-01 11:15:17|2021-01-01 11:24:55|         231|         148|
|2021-01-19 12:05:32|2021-01-19 12:33:46|         228|         210|
|2021-01-17 13:54:52|2021-01-17 14:07:03|          39|          61|
|2021-01-30 18:03:33|2021-01-30 18:23:17|          42|         250|
|2021-01-16 12:36:55|2021-01-16 13:03:23|         131|         265|
|2021-01-30 23:07:14|2021-01-30 23:27:34|       

In [38]:
df.dtypes

[('hvfhs_license_num', 'string'),
 ('dispatching_base_num', 'string'),
 ('pickup_datetime', 'timestamp'),
 ('dropoff_datetime', 'timestamp'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('SR_Flag', 'string')]

In [36]:
base_id('97')

'A-9710y'

## Disconnect Ngrok public endpoint

In [40]:
# Optionally put the tunnel down
ngrok.disconnect(public_url)

NameError: name 'ngrok' is not defined